In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -qU -r /content/drive/MyDrive/AiChan/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.

In [ ]:
import os
import faiss
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter)
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings # Hoặc langchain_openai tùy bạn dùng

def create_faiss_index(embeddings_model, index_type: str):
    """
    Tự động tạo FAISS index dựa trên tên thuật toán và kích thước vector của mô hình embeddings.
    """
    # Tạo 1 vector mẫu để lấy dimension (d)
    dummy_embedding = embeddings_model.embed_query("test dimension")
    d = len(dummy_embedding)

    if index_type == "IndexHNSWFlat":
        # M=32 là số lượng liên kết lân cận, có thể tuỳ chỉnh
        return faiss.IndexHNSWFlat(d, 32)
    elif index_type == "IndexFlatL2":
        return faiss.IndexFlatL2(d)
    elif index_type == "IndexFlatIP": # Inner Product (Tốt cho Cosine Similarity nếu vector được normalize)
        return faiss.IndexFlatIP(d)
    else:
        print(f"Chưa hỗ trợ {index_type} trực tiếp, mặc định dùng IndexFlatL2.")
        return faiss.IndexFlatL2(d)

def process_and_save_data(md_file_path: str,
                          save_dir: str,
                          headers_to_split_on: list,
                          embeddings_model_name: str,
                          chunk_size: int = 1000,
                          chunk_overlap: int = 150,
                          index_type: str = "IndexHNSWFlat"):
    """
    Đọc file Markdown, chia cắt văn bản, tạo FAISS DB với index custom và lưu vào Google Drive.
    """
    print(f"1. Đang đọc file từ: {md_file_path}")
    with open(md_file_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    print("2. Đang chia cắt văn bản theo Markdown Headers...")
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    md_header_splits = markdown_splitter.split_text(md_text)

    # Kỹ thuật bổ sung: Đảm bảo các chunk không quá lớn vượt context window
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    documents = text_splitter.split_documents(md_header_splits)
    print(f"-> Đã tạo {len(documents)} chunks.")

    print(f"3. Khởi tạo mô hình Embeddings: {embeddings_model_name}")
    # Đang sử dụng HuggingFace làm ví dụ, bạn có thể thay bằng OpenAIEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name=embeddings_model_name)

    print(f"4. Khởi tạo FAISS với thuật toán: {index_type}")
    index = create_faiss_index(embeddings, index_type)

    # Khởi tạo vector store rỗng với index custom
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={}
    )

    print("5. Đang embedding và đưa dữ liệu vào Vector DB...")
    vector_store.add_documents(documents)

    print(f"6. Lưu DB vào Google Drive tại: {save_dir}")
    os.makedirs(save_dir, exist_ok=True)
    vector_store.save_local(save_dir)
    print("-> Xong nhiệm vụ lưu trữ!\n")

    return save_dir, embeddings

def test_load_data(save_dir: str, embeddings_model, query: str = "Ngữ pháp N3"):
    """
    Load lại dữ liệu từ Google Drive để kiểm thử tính toàn vẹn và thực hiện tìm kiếm thử nghiệm.
    """
    print(f"--- BẮT ĐẦU KIỂM THỬ TỪ: {save_dir} ---")
    try:
        # allow_dangerous_deserialization=True là bắt buộc trong các phiên bản Langchain mới khi load file local
        loaded_vector_store = FAISS.load_local(
            folder_path=save_dir,
            embeddings=embeddings_model,
            allow_dangerous_deserialization=True,

        )
        print("-> Tải cơ sở dữ liệu thành công!")

        print(f"\nKiểm thử tìm kiếm với query: '{query}'")
        results = loaded_vector_store.similarity_search_with_score(query, k=2)

        for i, (doc, score) in enumerate(results):
            print(f"\nKết quả {i+1} (Khoảng cách: {score:.4f}):")
            print(f"Metadata (Headers): {doc.metadata}")
            print(f"Nội dung: {doc.page_content[:200]}...")

    except Exception as e:
        print(f"Lỗi trong quá trình tải hoặc truy vấn: {e}")

In [ ]:
MD_FILE_PATH = '/content/drive/MyDrive/AI_NARAGI/japanese/grammar/guide.md'
SAVE_DIR = '/content/drive/MyDrive/AI_NARAGI/faiss'
EMBEDDINGS_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_ALGORITHM = "IndexIVFPQ"


if __name__ == "__main__":
    # Cấu trúc header cần cắt
    HEADERS_TO_SPLIT_ON = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4")]
    # 1. Chạy tiền xử lý
    saved_path, embeddings_instance = process_and_save_data(
        md_file_path=MD_FILE_PATH,
        save_dir=SAVE_DIR,
        headers_to_split_on=HEADERS_TO_SPLIT_ON,
        embeddings_model_name=EMBEDDINGS_MODEL,
        index_type=INDEX_ALGORITHM)

In [ ]:
test_load_data(save_dir=saved_path,
               embeddings_model=embeddings_instance,
               query="V-masu + nagara")